<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset MNIST handwritten digits utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (MNIST handwritten digits)

- Utilize os arquivos `*.csv` disponibilizados via Google Classroom
- Use: `as_frame=False`
- Use: `mnist_784`

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `MNIST handwritten digits` (arquivos de treino e teste)
Realize a separação do conjunto de treino como treino e validação
Utilize `train_test_split` com controle de aleatoriedade (seed)
Retorne: `X_train`, `X_val`, `y_train`, `y_val`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

def load_data(seed):
    train_df = pd.read_csv("mnist_train.csv")

    X = train_df.drop(columns=["label"]).values
    y = train_df["label"].values

    X_train, X_val, y_train, y_val = train_test_split(
        X, y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )
    return X_train, X_val, y_train, y_val

In [ ]:

X_train, X_val, y_train, y_val = load_data(42)

print(f"Treino: {X_train.shape}")
print(f"Validação: {X_val.shape}")

Treino: (48000, 784)
Validação: (12000, 784)


Para o Random Forest, não é necessário normalizar os dados. Isso porque o modelo tome a decisão por meio de limiares, definindo se o valor é maior ou menor que determinado número. Assim, é analisado cada medida individualmente, não variando conforme escalas distintas.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

def train_random_forest(X_train, y_train, seed=42):
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    print("Random Forest treinado")
    return model

def train_decision_tree(X_train, y_train, seed=42):
    model = DecisionTreeClassifier(
        random_state=seed
    )
    model.fit(X_train, y_train)
    print("Árvore de Decisão treinada")
    return model


rf = train_random_forest(X_train, y_train, seed=42)
dt = train_decision_tree(X_train, y_train, seed=42)

Random Forest treinado
Árvore de Decisão treinada


# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [9]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)   
    accuracy = accuracy_score(y_test, y_pred)  
    return accuracy


acc = evaluate(rf, X_val, y_val)
print(f"Acurácia na validação: {acc:.4f}")

Acurácia na validação: 0.9665


**Adicione seu texto de solução aqui**.

A função evaluate realizou predições com o model.predict e comparou os os labels reais a partir do accuracy_score. A acurácia na validação indica que o modelo teve aproximadamente 96% de acerto.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [17]:
def run_pipeline(model_type="rf", seed=42):
    
    X_train, X_val, y_train, y_val = load_data(seed=seed)

    
    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed=seed)
    elif model_type == "dt":
        model = train_decision_tree(X_train, y_train, seed=seed)
    else:
        raise ValueError(f"Modelo '{model_type}' não suportado.")

    
    accuracy = evaluate(model, X_val, y_val)
    print(f"Acurácia na validação: {accuracy:.4f}")
    return accuracy, model


acc_rf, rf = run_pipeline(model_type="rf", seed=42)
acc_dt, dt = run_pipeline(model_type="dt", seed=42)

Random Forest treinado
Acurácia na validação: 0.9665
Árvore de Decisão treinada
Acurácia na validação: 0.8701


**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

Quando o  max_depth=None, a árvore cresce sem restrição até que cada folha contenha apenas um único exemplo de treino. Isso faz o modelo memorizar os dados em vez de aprender padrões gerais, resultando em 100% de acurácia no treino (comportamento de overfitting). A profundidade exata em que o overfitting começa a ocorrer não é mensurável, mas quanto maior a profundidade, maior a chance de overfitting.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest

## Apresente:
- Acurácia, Precisão, Recall e F1-Score para o modelo

**Solução**:

In [21]:
print("=== Random Forest ===")
evaluate_full(rf, X_val, y_val)

print("\n=== Árvore de Decisão ===")
evaluate_full(dt, X_val, y_val)

=== Random Forest ===
Acurácia : 0.9665
Precisão : 0.9665
Recall   : 0.9665
F1-Score : 0.9665

=== Árvore de Decisão ===
Acurácia : 0.8701
Precisão : 0.8702
Recall   : 0.8701
F1-Score : 0.8701


# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [22]:
for seed in [42, 7]:
    print(f"\n--- Seed {seed} ---")
    print("Random Forest:")
    acc_rf, rf_temp = run_pipeline(model_type="rf", seed=seed)
    print("Árvore de Decisão:")
    acc_dt, dt_temp = run_pipeline(model_type="dt", seed=seed)


--- Seed 42 ---
Random Forest:
Random Forest treinado
Acurácia na validação: 0.9665
Árvore de Decisão:
Árvore de Decisão treinada
Acurácia na validação: 0.8701

--- Seed 7 ---
Random Forest:
Random Forest treinado
Acurácia na validação: 0.9689
Árvore de Decisão:
Árvore de Decisão treinada
Acurácia na validação: 0.8710


Executando o pipeline com seeds 42 e 7, obtivemos os seguintes resultados:
- Random Forest: 96,65% (seed 42) e 96,89% (seed 7)
- Árvore de Decisão: 87,01% (seed 42) e 87,10% (seed 7)

O experimento é reprodutível pois foi fixado random_state tanto no train_test_split e no RandomForestClassifier. Isso garante que qualquer pessoa que rode o código com a mesma seed obtenha exatamente os mesmos resultados. A variação entre seeds diferentes é pequena nos dois modelos, indicando que ambos são estáveis e não dependem de uma divisão específica dos dados para funcionar bem.

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [27]:


test_df = pd.read_csv("mnist_test.csv")
X_test = test_df.drop(columns=["label"]).values
y_test = test_df["label"].values


acc_rf_train = evaluate(rf, X_train, y_train)
acc_rf_test  = evaluate(rf, X_test, y_test)

acc_dt_train = evaluate(dt, X_train, y_train)
acc_dt_test  = evaluate(dt, X_test, y_test)

print("=== Random Forest ===")
print(f"Treino : {acc_rf_train:.4f}")
print(f"Teste  : {acc_rf_test:.4f}")
print(f"Gap    : {acc_rf_train - acc_rf_test:.4f}")

print("\n=== Árvore de Decisão ===")
print(f"Treino : {acc_dt_train:.4f}")
print(f"Teste  : {acc_dt_test:.4f}")
print(f"Gap    : {acc_dt_train - acc_dt_test:.4f}")

=== Random Forest ===
Treino : 1.0000
Teste  : 0.9673
Gap    : 0.0327

=== Árvore de Decisão ===
Treino : 1.0000
Teste  : 0.8724
Gap    : 0.1276


Ambos os modelos atingiram 100% de acurácia no treino, pois, devido à falta de restrição de profundidade, os modelos memorizam os dados de treino. Porém, o comportamento na validação foi diferente:

Random Forest: diferença de 3,27% entre treino e validação, apontando um overfitting leve, pois o bagging combina várias árvores e reduz a variância do modelo final.
Árvore de Decisão: diferença de 12,76%, apontando um overfitting alto, pois uma única árvore sem restrição memoriza os dados.

Portanto, a Árvore de Decisão sofre muito mais overfitting que o Random Forest.

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`

## Analise:
- O desempenho muda significativamente?

In [29]:
n_estimators_list = [10, 25, 50, 100, 200]

for n in n_estimators_list:
    model = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    acc_train = evaluate(model, X_train, y_train)
    acc_test  = evaluate(model, X_test, y_test)

    print(f"n_estimators={n:4d} | Treino: {acc_train:.4f} | Teste: {acc_test:.4f}")

n_estimators=  10 | Treino: 0.9992 | Teste: 0.9439
n_estimators=  25 | Treino: 0.9999 | Teste: 0.9576
n_estimators=  50 | Treino: 1.0000 | Teste: 0.9656
n_estimators= 100 | Treino: 1.0000 | Teste: 0.9673
n_estimators= 200 | Treino: 1.0000 | Teste: 0.9681


O desempenho muda significativamente até 50 árvores, depois estabiliza com variações muito discretas. O treino fica em 100% em todos os casos.

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

1.
Não necessariamente. A acurácia é uma métrica agregada que pode mascarar problemas em datasets desbalanceados. No dataset em questão, as métricas de acurácia, precisão, recall e F1-Score estavam equivalentes.

2.
Fixando random_state em todas as etapas aleatórias, garantindo que qualquer pessoa reproduza exatamente os mesmos resultados. Além disso, foi testadas seeds diferentes e a variação foi pequena, indicando que o modelo é estável e não depende de uma divisão favorável dos dados.

3.
Usar o conjunto de teste para escolher hiperparâmetros (Q8) pode contaminar a avaliação final. Além disso, a ausência de validação cruzada limita a confiabilidade dos resultados, pois avaliamos o modelo em apenas uma divisão dos dados.

4.
O pipeline é razoavelmente confiável pois garante reprodutibilidade com controle de seed, separa corretamente treino de teste antes do treinamento, e usa stratify para preservar a distribuição de classes. A principal limitação é a ausência de validação cruzada, que tornaria a estimativa de desempenho mais robusta.
